# 02 · Distances: from raw attributes to a distance matrix

**Lecture 1 · Block B4 · Lab · 25 minutes (if time is short: Task 3 only)**

**Learning goal:** We turn attributes of different types into comparable numbers, implement the Minkowski and the cosine distance, and compute a Gower distance matrix for our dataset.

**What we do:**
1. Task 1 - `minkowski` and `cosine_distance` (easy, about 4 min)
2. Task 2 - `preprocess`: scaling and encoding (medium, about 6 min)
3. Task 3 - `gower_distance` (**the core task**, about 10 min). It does not need Tasks 1-2: if time is short, start here.

Below every task there is a check cell that prints `PASS`, `FAIL` (with a reason) or `TODO`. At the end the feature matrix `X` and the distance matrix `D` are saved for `03_clustering.ipynb`.

In [ ]:
DATASET = "penguins"  # or "wholesale", "auto_mpg", "students" (see 01_dataset.ipynb)

In [ ]:
from collections.abc import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.spatial import distance as scipy_distance

import ddm

## Setup: data and attribute types (given)

In [ ]:
df = ddm.load_dataset(DATASET)
info = ddm.dataset_info(DATASET)

TYPES = ddm.load_json(DATASET, "types")  # saved by 01_dataset.ipynb
if TYPES is None:
    TYPES = {
        col: "numeric"
        if pd.api.types.is_numeric_dtype(df[col]) and col not in info.label_candidates
        else "ignore"
        for col in df.columns
    }
    print("NOTICE: no TYPES from 01_dataset.ipynb found. Using a BASELINE: numeric columns -> 'numeric', all others -> 'ignore'.")
    print("        Run 01_dataset.ipynb to the end to use your own classification.")
pd.Series(TYPES, name="type").to_frame().T

### Missing values and skewed attributes (given)

Two small decisions come before any distance:

1. Objects where less than half of the used attributes are known are dropped. The remaining
   gaps are handled inside `preprocess` and `gower_distance`.
2. Strongly right-skewed attributes (a few huge values, e.g. spending) are log-transformed
   with log(1 + x). Rule of thumb: skewness above 2.

In [ ]:
def keep_rows(df: pd.DataFrame, types: dict[str, str], min_present: float = 0.5) -> pd.DataFrame:
    """Drop objects where less than ``min_present`` of the used attributes are known."""
    used = [c for c, t in types.items() if t != "ignore"]
    share_present = df[used].notna().mean(axis=1)
    return df.loc[share_present >= min_present]

def log_transform(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Return a copy of df with log(1 + x) applied to the given columns."""
    out = df.copy()
    for col in columns:
        out[col] = np.log1p(out[col].astype(float))
    return out


used = [c for c, t in TYPES.items() if t != "ignore"]
numeric_used = [c for c in used if TYPES[c] == "numeric"]
print("Skewness of the numeric attributes (above 2: consider a log transform):")
print(df[numeric_used].skew().round(2).to_string())

In [ ]:
# Our choice: which columns get log(1 + x)? For "wholesale" all six spending columns are candidates.
LOG_COLUMNS = []

data = log_transform(keep_rows(df, TYPES), LOG_COLUMNS)
print(f"{len(df) - len(data)} of {len(df)} objects dropped (less than half of the used attributes known), {len(data)} remain.")

### Why scaling matters (given)

The share of the total variance per numeric attribute, on the raw values. A raw Euclidean
distance is dominated by the attributes with the largest variance.

In [ ]:
raw = data[numeric_used].astype(float)
variance_share = raw.var() / raw.var().sum() * 100
print("Share of the total variance in % (raw values):")
print(variance_share.round(2).to_string())

## Task 1 · Minkowski and cosine distance (easy, about 4 minutes)

For two objects $x, y \in \mathbb{R}^m$:

$$d_q(x, y) = \Big(\sum_{i=1}^{m} |x_i - y_i|^q\Big)^{1/q}
\qquad q = 1: \text{Manhattan}, \quad q = 2: \text{Euclidean}, \quad q \to \infty: \max_i |x_i - y_i| \ \text{(Chebyshev)}$$

$$d_{\cos}(x, y) = 1 - \frac{x \cdot y}{\lVert x \rVert \, \lVert y \rVert}$$

The cosine distance only looks at the *direction* of the vectors, not at their length.

In [ ]:
def minkowski(x: np.ndarray, y: np.ndarray, q: float = 2) -> float:
    """Minkowski distance of order q between two vectors.

    Parameters
    ----------
    x, y : 1D arrays of the same length m
    q : order; 1 = Manhattan, 2 = Euclidean, ``np.inf`` = Chebyshev (maximum)

    Returns
    -------
    float
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    # TODO: absolute differences |x_i - y_i| (one numpy line), then
    #       return (sum of diff ** q) ** (1 / q).
    #       For q = np.inf return the largest absolute difference instead (test with np.isinf(q)).
    raise NotImplementedError("minkowski")

In [ ]:
def cosine_distance(x: np.ndarray, y: np.ndarray) -> float:
    """Cosine distance = 1 - cosine similarity of two vectors.

    Parameters
    ----------
    x, y : 1D arrays of the same length, not all zeros

    Returns
    -------
    float in [0, 2]: 0 = same direction, 1 = orthogonal, 2 = opposite direction
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    # TODO: dot product x @ y, lengths np.linalg.norm(x) and np.linalg.norm(y)
    raise NotImplementedError("cosine_distance")

In [ ]:
# Check Task 1 (compares with scipy.spatial.distance)
def test_minkowski():
    assert np.isclose(minkowski([0, 0], [3, 4], 2), 5.0), "Euclidean distance of (0, 0) and (3, 4) must be 5"
    assert np.isclose(minkowski([0, 0], [3, 4], 1), 7.0), "Manhattan distance of (0, 0) and (3, 4) must be 7"
    rng = np.random.default_rng(0)
    x, y = rng.normal(size=6), rng.normal(size=6)
    for q in (1, 2, 3):
        got, expected = minkowski(x, y, q), scipy_distance.minkowski(x, y, p=q)
        assert np.isclose(got, expected), f"q={q}: got {got:.4f}, scipy says {expected:.4f}"
    got, expected = minkowski(x, y, np.inf), scipy_distance.chebyshev(x, y)
    assert np.isclose(got, expected), f"q=np.inf: got {got:.4f}, expected the largest difference {expected:.4f}"


def test_cosine():
    assert np.isclose(cosine_distance([1, 0], [0, 1]), 1.0), "orthogonal vectors must have distance 1"
    assert np.isclose(cosine_distance([1, 2], [2, 4]), 0.0), "same direction must give 0 (length does not matter)"
    rng = np.random.default_rng(1)
    x, y = rng.normal(size=6), rng.normal(size=6)
    got, expected = cosine_distance(x, y), scipy_distance.cosine(x, y)
    assert np.isclose(got, expected), f"got {got:.4f}, scipy says {expected:.4f}"


ddm.check("minkowski", test_minkowski)
ddm.check("cosine_distance", test_cosine)

## Task 2 · `preprocess`: scaling and encoding (medium, about 6 minutes)

Most algorithms need a purely numeric matrix $X$ (objects × features). Per attribute type:

| Type | Missing values | Transformation | Result |
|------|----------------|----------------|--------|
| numeric, ordinal | median | z-score $z = (x - \bar{x}) / s$ | 1 column, mean 0, std 1 |
| binary | most frequent value | 1 for one value, 0 for the other | 1 column `col=value` |
| nominal | most frequent value | one-hot: one 0/1 column per category | k columns `col=value` |
| ignore | | dropped | |

The loop and the nominal branch are given as an example. Our part: the numeric/ordinal branch
and the binary branch. Replace each `raise NotImplementedError(...)` by your code.

<details>
<summary><b>Hint</b> (click to open)</summary>

- `s.fillna(s.median())` fills the gaps of a Series `s`.
- `s.mean()`, `s.std()` give mean and standard deviation.
- `(s == one).astype(float)` gives a 0/1 Series.
- Store each new column in the dict: `features["name"] = series`.

</details>

In [ ]:
def preprocess(df: pd.DataFrame, types: dict[str, str]) -> pd.DataFrame:
    """Turn the raw attributes into a numeric feature matrix.

    Parameters
    ----------
    df : DataFrame
        The raw data, one row per object (missing values allowed).
    types : dict
        ``{column: "numeric" | "ordinal" | "nominal" | "binary" | "ignore"}``.

    Returns
    -------
    DataFrame
        Same index as ``df``, only numbers, no missing values:

        * numeric, ordinal: missing -> median, then z-score ``(x - mean) / std``
        * binary: missing -> most frequent value, then one 0/1 column ``col=value``
        * nominal: missing -> most frequent value, then one 0/1 column per category (one-hot)
        * ignore: dropped
    """
    features = {}  # new column name -> pandas Series
    for col, kind in types.items():
        s = df[col]
        if kind == "ignore":
            continue
        if kind in ("numeric", "ordinal"):
            if kind == "ordinal" and not pd.api.types.is_numeric_dtype(s):
                s = s.map({v: r for r, v in enumerate(sorted(s.dropna().unique()))})  # text -> rank
            s = s.astype(float)
            # TODO (a): fill the missing values with the median
            # TODO (b): z-score and store it: features[col] = (s - mean) / std
            #           (if std == 0 the column is constant: store s * 0.0)
            raise NotImplementedError("preprocess: numeric/ordinal branch")
        elif kind == "binary":
            s = s.fillna(s.mode().iloc[0])
            one = sorted(s.unique())[-1]  # the value coded as 1, e.g. "yes" or "male"
            # TODO (c): store a 0/1 column named f"{col}={one}": 1.0 where s == one, else 0.0
            raise NotImplementedError("preprocess: binary branch")
        elif kind == "nominal":
            # given as an example: one-hot encoding, one 0/1 column per category
            s = s.fillna(s.mode().iloc[0])
            for value in sorted(s.unique()):
                features[f"{col}={value}"] = (s == value).astype(float)
        else:
            raise ValueError(f"Unknown type '{kind}' for column '{col}'")
    return pd.DataFrame(features, index=df.index)

In [ ]:
# Check Task 2 on a tiny example
def test_preprocess():
    toy = pd.DataFrame({
        "length": [1.0, 2.0, np.nan, 5.0],
        "colour": ["red", "blue", "red", np.nan],
        "member": ["no", "yes", "yes", "no"],
        "id": [101, 102, 103, 104],
    })
    toy_types = {"length": "numeric", "colour": "nominal", "member": "binary", "id": "ignore"}
    out = preprocess(toy, toy_types)
    assert isinstance(out, pd.DataFrame), "preprocess must return a DataFrame"
    assert out.shape == (4, 4), f"expected 4 columns (length, colour=blue, colour=red, member=yes), got {list(out.columns)}"
    assert not out.isna().any().any(), "there are still missing values"
    z = out["length"].to_numpy()
    filled = np.array([1.0, 2.0, 2.0, 5.0])  # the gap is filled with the median 2.0
    z_expected = [(filled - filled.mean()) / filled.std(ddof=d) for d in (1, 0)]  # pandas or numpy std
    assert any(np.allclose(z, e) for e in z_expected), \
        f"length: expected z-scores {np.round(z_expected[0], 3)} (fill with the median first), got {np.round(z, 3)}"
    member = out[[c for c in out.columns if c.startswith("member")][0]]
    assert list(member) == [0, 1, 1, 0], f"binary: 'yes' -> 1, 'no' -> 0, got {list(member)}"
    assert "id" not in out.columns, "'ignore' columns must be dropped"


ddm.check("preprocess", test_preprocess)

Now on our data:

In [ ]:
try:
    features = preprocess(data, TYPES)
    X = features.to_numpy(dtype=float)
    print(f"Feature matrix X: {X.shape[0]} objects x {X.shape[1]} features")
    display(features.describe().T.round(2))
except NotImplementedError as err:
    features, X = None, None
    print(f"NOTICE: preprocess is not implemented yet ({err}). X is skipped for now.")

### Distance matrix with our Minkowski distance (given)

`distance_matrix` fills an $n \times n$ matrix with a plain double loop. We compare the
nearest neighbours of one object on the raw values and on the z-scored features.

In [ ]:
def distance_matrix(X: np.ndarray, metric: Callable[[np.ndarray, np.ndarray], float]) -> np.ndarray:
    """Pairwise distance matrix with a plain double loop (uses symmetry)."""
    X = np.asarray(X, dtype=float)
    n = X.shape[0]
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            d = metric(X[i], X[j])
            D[i, j] = d
            D[j, i] = d
    return D


def show_neighbours(D: np.ndarray, i: int, k: int = 3) -> pd.DataFrame:
    """Object i and its k nearest neighbours (original attribute values)."""
    order = np.argsort(D[i])[: k + 1]  # the object itself comes first (distance 0)
    shown = df.loc[data.index[order]].copy()
    shown.insert(0, "distance", D[i, order].round(3))
    return shown

In [ ]:
OBJECT = 0  # position of the object we look at

try:
    raw_values = data[numeric_used].astype(float)
    raw_values = raw_values.fillna(raw_values.median()).to_numpy()
    D_raw = distance_matrix(raw_values, lambda a, b: minkowski(a, b, 2))
    print("Nearest neighbours, Euclidean distance on the RAW numeric values:")
    display(show_neighbours(D_raw, OBJECT))
    if X is not None:
        D_euclid = distance_matrix(X, lambda a, b: minkowski(a, b, 2))
        print("Nearest neighbours, Euclidean distance on the preprocessed features X:")
        display(show_neighbours(D_euclid, OBJECT))
except NotImplementedError as err:
    print(f"NOTICE: {err} is not implemented yet - comparison skipped.")

## Task 3 · Gower distance (the core task, about 10 minutes)

Real data mixes types. The Gower distance compares every attribute $k$ on its own, with a
value in $[0, 1]$, and averages:

$$d_{\text{Gower}}(i, j) = \frac{\sum_k \delta_{ijk}\, d_{ijk}}{\sum_k \delta_{ijk}}
\qquad
d_{ijk} = \begin{cases}
\dfrac{|x_{ik} - x_{jk}|}{R_k} & \text{numeric, ordinal } (R_k = \max - \min \text{ of attribute } k)\\[2mm]
0 \text{ if } x_{ik} = x_{jk}, \text{ else } 1 & \text{nominal, binary}
\end{cases}$$

$\delta_{ijk} = 0$ if $x_{ik}$ or $x_{jk}$ is missing, else 1. So missing values are simply
left out of the average for that pair.

**Plan:** for every used attribute we build an $n \times n$ matrix `part` with all $d_{ijk}$
(NaN where a value is missing). `np.nanmean` over all these matrices then *is* the formula above,
including the $\delta$. The loop and the final average are given; we fill in the two `part`
computations.

<details>
<summary><b>Hint:</b> all pairwise differences at once (click to open)</summary>

```python
v = np.array([1.0, 3.0, 6.0])
v[:, None] - v[None, :]   # column minus row -> 3 x 3 matrix
# array([[ 0., -2., -5.],
#        [ 2.,  0., -3.],
#        [ 5.,  3.,  0.]])
```
The same works with `!=` for categories: `(v[:, None] != v[None, :])` is True where two objects differ.
NaN in a difference stays NaN, which is exactly what we want.
</details>

<details>
<summary><b>Hint:</b> the small test example by hand (click to open)</summary>

Three objects: size = 1, 3, 5 (range 4); colour = red, red, blue; flag = yes, *missing*, no.

- d(0,1): size 2/4 = 0.5, colour 0, flag unknown → (0.5 + 0) / 2 = **0.25**
- d(0,2): size 4/4 = 1, colour 1, flag 1 → 3 / 3 = **1.0**
- d(1,2): size 2/4 = 0.5, colour 1, flag unknown → 1.5 / 2 = **0.75**
</details>

In [ ]:
def gower_distance(df: pd.DataFrame, types: dict[str, str]) -> np.ndarray:
    """Gower distance matrix for attributes of mixed types.

    Parameters
    ----------
    df : DataFrame
        Raw data (NOT z-scored, NOT one-hot encoded), missing values allowed.
    types : dict
        ``{column: "numeric" | "ordinal" | "nominal" | "binary" | "ignore"}``.

    Returns
    -------
    D : array of shape (n, n)
        ``D[i, j]`` = average over the used attributes (known for both objects) of

        * numeric, ordinal: ``|x_i - x_j| / range`` of the column
        * nominal, binary: 0 if equal, 1 if different
    """
    parts = []  # one n x n matrix per used attribute, values in [0, 1], NaN = unknown
    for col, kind in types.items():
        if kind == "ignore":
            continue
        s = df[col]
        if kind in ("numeric", "ordinal"):
            if kind == "ordinal" and not pd.api.types.is_numeric_dtype(s):
                s = s.map({v: r for r, v in enumerate(sorted(s.dropna().unique()))})  # text -> rank
            v = s.to_numpy(dtype=float)  # NaN marks a missing value
            # TODO (a): value_range = largest minus smallest value (np.nanmax and np.nanmin ignore NaN)
            # TODO (b): diff = all pairwise |v_i - v_j| at once (n x n): see the hint above
            # TODO (c): part = diff / value_range   (if value_range == 0: part = diff * 0.0)
            raise NotImplementedError("gower_distance: numeric/ordinal part")
        elif kind in ("nominal", "binary"):
            v = s.to_numpy(dtype=object)
            missing = s.isna().to_numpy()  # boolean mask of the missing values
            # TODO (d): part = 1.0 where v_i != v_j, 0.0 where equal (same trick, then .astype(float))
            # TODO (e): set the rows AND the columns of the missing objects to np.nan
            raise NotImplementedError("gower_distance: nominal/binary part")
        else:
            raise ValueError(f"Unknown type '{kind}' for column '{col}'")
        parts.append(part)
    D = np.nanmean(np.stack(parts), axis=0)  # average over attributes known for both objects
    np.fill_diagonal(D, 0.0)
    return D

In [ ]:
# Check Task 3
def test_gower_small():
    toy = pd.DataFrame({
        "size": [1.0, 3.0, 5.0],
        "colour": ["red", "red", "blue"],
        "flag": ["yes", np.nan, "no"],
    })
    toy_types = {"size": "numeric", "colour": "nominal", "flag": "binary"}
    expected = np.array([[0.0, 0.25, 1.0], [0.25, 0.0, 0.75], [1.0, 0.75, 0.0]])
    got = gower_distance(toy, toy_types)
    assert np.shape(got) == (3, 3), f"D must be 3 x 3, got shape {np.shape(got)}"
    assert np.allclose(got, expected), f"expected\n{expected}\ngot\n{np.round(got, 3)}"


def test_gower_on_data():
    D_test = gower_distance(data, TYPES)
    n = len(data)
    assert D_test.shape == (n, n), f"D must be {n} x {n}"
    assert not np.isnan(D_test).any(), "D contains NaN"
    assert np.allclose(D_test, D_test.T), "D must be symmetric"
    assert np.allclose(np.diag(D_test), 0), "the diagonal must be 0"
    assert D_test.min() >= 0 and D_test.max() <= 1 + 1e-12, "Gower distances must lie in [0, 1]"


ddm.check("gower_distance, small example by hand", test_gower_small)
ddm.check("gower_distance, properties on our data", test_gower_on_data)

Now on our data:

In [ ]:
try:
    D = gower_distance(data, TYPES)
    print(f"Gower distance matrix D: {D.shape}, values from {D.min():.3f} to {D.max():.3f}")
except NotImplementedError as err:
    D = None
    print(f"NOTICE: gower_distance is not implemented yet ({err}). D is skipped for now.")

### Sanity checks (given)

A distance matrix should be symmetric with a zero diagonal. A *metric* also fulfils the
triangle inequality $d(i, k) \le d(i, j) + d(j, k)$; we test it on random triples.
And the most important check: do the nearest neighbours make sense to a human?

In [ ]:
def sanity_checks(D: np.ndarray, n_triples: int = 20000, seed: int = 0) -> None:
    rng = np.random.default_rng(seed)
    i, j, k = rng.integers(0, len(D), size=(3, n_triples))
    violations = int((D[i, k] > D[i, j] + D[j, k] + 1e-12).sum())
    print(f"symmetric:            {np.allclose(D, D.T)}")
    print(f"zero diagonal:        {np.allclose(np.diag(D), 0)}")
    print(f"range:                {D.min():.3f} .. {D.max():.3f}")
    print(f"triangle inequality:  {violations} violations in {n_triples} random triples")


if D is None:
    print("NOTICE: no D yet (Task 3) - sanity checks skipped.")
else:
    sanity_checks(D)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(D[np.triu_indices_from(D, k=1)], bins=60, color="#2a6fdb")
    ax.set_xlabel("Gower distance")
    ax.set_ylabel("number of pairs")
    ax.set_title(f"All pairwise Gower distances ({DATASET})")
    plt.show()
    print(f"Nearest neighbours of object {OBJECT} (Gower):")
    display(show_neighbours(D, OBJECT))

## Save the results for 03_clustering (given)

`03_clustering.ipynb` and `04_beyond.ipynb` load `X`, `D` and the row index from
`data/processed/`. If something is missing there, they fall back to a simple baseline.

In [ ]:
meta = {
    "dataset": DATASET,
    "rows": data.index.tolist(),  # which objects X and D describe, in this order
    "types": TYPES,
    "log_columns": LOG_COLUMNS,
    "feature_names": None if features is None else list(features.columns),
}
ddm.save_json(DATASET, "meta", meta)
if X is not None:
    print("Saved", ddm.save_array(DATASET, "X", X))
else:
    print("NOT saved: X (Task 2 missing)")
if D is not None:
    print("Saved", ddm.save_array(DATASET, "D", D))
else:
    print("NOT saved: D (Task 3 missing)")

## Reflection (2 minutes)

1. Look at the histogram of the Gower distances. Why can there be "steps" or clusters of
   distance values when the data has nominal or binary attributes?
2. Raw vs. scaled nearest neighbours: which attribute decided the raw neighbours?
3. Gower gives every attribute the same weight. Is that what we want for our question?